# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — "What Predicts Health?" (Random Forest, page 27).** The paper reports Average
Position (43%) and Impressions (32%) as the top two importances for a Random Forest predicting
Health Score, holdout-tested. The paper's own caveat is exactly right: *"the target itself is
partly constructed from some of these inputs, so importance is descriptive rather than causal."*
Health Score is disclosed on page 5 and again on page 36 as `Impressions (30 pts) + position (30
pts) + CTR (20 pts) + scroll depth (20 pts)`  a formula, not an independent outcome.

**My methodology question:** Average Position + Impressions are 2 of the label's 4 ingredients,
worth 60% of the formula's own point weight  and they're also the top 2 "predicted" features,
together carrying 75% of the model's importance (43% + 32%). The 80/20 holdout protects against
*overfitting noise*  it confirms the relationship holds on rows the model didn't train on. It
does nothing to protect against *construct overlap*  the label being partly made of the same
material as the inputs — because that overlap was baked in at label-definition time, before any
split happened. A holdout split can't rescue a target that's circular by construction. Would the
paper's existing caveat land more concretely for a reader if it named the overlap directly (e.g.
"Position and Impressions alone are 60% of this label's own formula") rather than the more
general "descriptive rather than causal"? Not a claim the paper got this wrong — it's already
disclosed — just a question about how far the disclosure goes.

**Finding B — "What Predicts Growth?" (Logistic Regression, page 29).** 71% holdout accuracy on an
80/20 split (split type confirmed on page 36's Methodology page), separating growing from
declining pages.

**My methodology questions, both about whether the validation design carries the claim:**

1. *Base rate.* Finding #1 (page 6) reports 74.8K growing vs. 45.6K declining pages across the
   full portfolio — 62.1% growing. If the ML-appendix sample (61.8K active-content rows) has a
   similar split, a trivial "always predict growing" rule would already land near 62% without
   learning anything, which would make 71% roughly **9 points of actual lift, not 71.** The paper
   doesn't report the base rate for this specific classifier's own sample anywhere I could find —
   which is itself the gap: a reader can't check this independently without it stated directly,
   the way this exact 71%-vs-62% shape is precisely what `hunting-leakage-and-validating` warns
   about by name.
2. *Split design.* The Methodology page states "80/20 split" for Random Forest, Logistic
   Regression, and the Decision Tree, but never says whether it's grouped by brand. The dataset
   spans 57 brands across ~61.8K ML-appendix rows — over 1,000 content pieces per brand on
   average. I didn't have to guess whether that matters: I checked it directly on my own Week-5
   model, same style of features, same style of 80/20 split, and a naive row-level split
   understated my held-out error by roughly **2x** relative to a split grouped by client (Section
   2 below reproduces that exact check). The paper's methodology page doesn't say which kind of
   split it used, so a reader has no way to know whether 71% would hold under a brand-held-out
   split, or drop the way mine did.

Both questions come from the same place: the paper is honest about the things it discloses (it
even names Health Score's formula and states the split type), the gap is in what's *not* stated —
the base rate for this specific sample, and whether "80/20" means row-level or grouped. That's a
disclosure gap, not a wrong-methodology accusation, and it's exactly the standard Section 2
holds this notebook's own model to next.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Receipts: pull the exact source pages directly from the PDF in this repo.
# Self-contained -- clones/chdirs into the repo if this session hasn't
# already done so, so this cell doesn't depend on an earlier setup cell
# having been run in the current runtime.
import os

REPO_DIR = "/content/ML-Assignement01"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/Abdulm111/ML-Assignement01.git {REPO_DIR}
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader("docs/flyrank-seo-research-march-2026.pdf")

def pdf_page(page_number):
    return reader.pages[page_number - 1].extract_text()

print("=== Page 27 -- Random Forest feature importance for Health Score ===")
print(pdf_page(27))

print("\n=== Page 29 -- Logistic Regression growth/decline classifier ===")
print(pdf_page(29))

print("\n=== Page 36 -- Methodology (health score formula, 80/20 split, confounders) ===")
print(pdf_page(36))

cwd: /content/ML-Assignement01
=== Page 27 -- Random Forest feature importance for Health Score ===
FlyRank ML APPENDIX — FEATURE IMPORTANCE
What Predicts Health?
Random Forest feature importance for predicting health score. The model is holdout-tested, but the target 
itself is partly constructed from some of these inputs, so importance is descriptive rather than causal.
FEATURE IMPORTANCE (RANDOM FOREST ’ HEALTH SCORE)
Average Position 43
Impressions 32
Scroll Depth 15
CTR 8
Clicks 2
Sessions 0
Content Age 0
Word Count 0
Days Visible 0
AI Sessions 0
Chart read: Higher bars mean the model leaned more heavily on that feature when estimating health score. Because health score already includes some visibility inputs, 
read this as model behavior, not as a standalone optimization order.
Average Position is the #1 predictor of health score at 43% importance, followed by Impressions (32%) and 
Scroll Depth (15%). This ranking shows which features the model uses most, though note that health

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Grouped, not time-aware — same reasoning as w05, restated because it now matters twice over.**
This is still a single-month cross-section (March 2026) predicting a same-month outcome, so
there's no "later" to hold out for a time split yet. Grouped by `client_hash_id` is the honest
design for the same reason it was in w05: rows from one client share hidden character that a
random split would let leak across train and test.

**Before/after, on the exact same modeling panel.** The pipeline below is identical to w05's (same
filters, same features, same frozen tier-only reference baseline) so the *only* thing that changes
between the two rows of the table is which split produced train/test. "Before" is a naive
row-level 80/20 split — the kind of default a first pass reaches for, and structurally the same
kind of split Section 1 just flagged as unstated in the FlyRank paper's own methodology page.
"After" is the grouped-by-client split w05 actually used. Same model (Random Forest, same
hyperparameters as w05's chosen model), same metrics, same frozen baseline — so any gap between
the two rows is attributable to the split, not to anything else changing underneath it.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

con.execute(f"""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")

gsc_monthly = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_month
    FROM march_facts WHERE gsc_data_available IS TRUE GROUP BY 1, 2
""").df()

ga4_monthly = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_month,
           SUM(sessions_ai) AS sessions_ai_month,
           SUM(scroll_events) AS scroll_events_month,
           COUNT(*) AS ga4_days_available
    FROM march_facts WHERE ga4_data_available IS TRUE GROUP BY 1, 2
""").df()

monthly = gsc_monthly.merge(ga4_monthly, on=["client_hash_id", "content_hash_id"], how="left")
monthly["has_ga4_month"] = monthly["ga4_days_available"].notna().astype(int)
for c in ["ga4_engaged_sessions_month", "sessions_ai_month", "scroll_events_month"]:
    monthly[c] = monthly[c].fillna(0)
monthly = monthly.drop(columns=["ga4_days_available"])
monthly["position_tier"] = np.select(
    [monthly["avg_position_month"] <= 3, monthly["avg_position_month"] <= 10,
     monthly["avg_position_month"] <= 20, monthly["avg_position_month"] <= 50],
    ["top_3", "page_1", "striking", "page_3_5"], default="deep"
)

content = con.sql(f"""
    SELECT content_hash_id, content_type, is_published, is_deleted
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

q = monthly.merge(content, on="content_hash_id", how="inner")
q = q[(q["is_published"] == True) & (q["is_deleted"] == False)]
q = q[(q["impressions_month"] >= 100) & (q["avg_position_month"] >= 1)].copy()
q["ctr"] = q["clicks_month"] / q["impressions_month"]

print(f"Modeling panel: {len(q)} rows, {q['client_hash_id'].nunique()} clients (matches w05)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling panel: 100702 rows, 44 clients (matches w05)


In [12]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

FEATURE_COLS = ["avg_position_month", "impressions_month", "ga4_engaged_sessions_month",
                 "sessions_ai_month", "scroll_events_month", "has_ga4_month"]

def make_X(df, ref_cols=None):
    X = pd.get_dummies(df[FEATURE_COLS + ["content_type"]], columns=["content_type"])
    return X.reindex(columns=ref_cols, fill_value=0) if ref_cols is not None else X

# Frozen tier-only reference, computed once on the full panel -- the SAME
# baseline is used for both rows of the before/after table below.
tier_mean_full = q.groupby("position_tier")["ctr"].mean()
global_mean_full = q["ctr"].mean()

def evaluate_split(train, test, label):
    train_X = make_X(train)
    test_X = make_X(test, train_X.columns)
    rf = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                random_state=42, n_jobs=-1)
    rf.fit(train_X, train["ctr"])
    pred = rf.predict(test_X)
    tier_pred = test["position_tier"].map(tier_mean_full).fillna(global_mean_full)
    tier_mse = mean_squared_error(test["ctr"], tier_pred)
    mse = mean_squared_error(test["ctr"], pred)
    return {
        "split": label,
        "test_rows": len(test),
        "test_clients": test["client_hash_id"].nunique(),
        "r2_vs_test_mean": round(r2_score(test["ctr"], pred), 4),
        "mae_ctr_pts": round(mean_absolute_error(test["ctr"], pred), 4),
        "pct_variance_reduction_vs_tier_baseline": round((1 - mse / tier_mse) * 100, 2),
    }

# BEFORE -- naive row-level random split
r_train, r_test = train_test_split(q, test_size=0.2, random_state=42)
overlap = set(r_train["client_hash_id"]) & set(r_test["client_hash_id"])
print(f"Clients shared between train/test under the naive random split: "
      f"{len(overlap)} of {q['client_hash_id'].nunique()}")

# AFTER -- grouped by client (what w05 actually used)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
g_train_idx, g_test_idx = next(gss.split(q, groups=q["client_hash_id"]))
g_train, g_test = q.iloc[g_train_idx].copy(), q.iloc[g_test_idx].copy()
g_overlap = set(g_train["client_hash_id"]) & set(g_test["client_hash_id"])
print(f"Clients shared between train/test under the grouped split: {len(g_overlap)} (must be 0)")

before_after = pd.DataFrame([
    evaluate_split(r_train, r_test, "BEFORE -- naive random split"),
    evaluate_split(g_train, g_test, "AFTER -- grouped by client"),
])
before_after

Clients shared between train/test under the naive random split: 37 of 44
Clients shared between train/test under the grouped split: 0 (must be 0)


,split,test_rows,test_clients,r2_vs_test_mean,mae_ctr_pts,pct_variance_reduction_vs_tier_baseline
0,BEFORE -- naive random split,20141,38,0.1880,0.0022,15.67
1,AFTER -- grouped by client,7009,9,0.1449,0.0032,15.08


In [13]:
# R^2/MAE favor the naive split, but %-variance-reduction favors the grouped
# split -- that's a ratio metric, so check what the denominator (the frozen
# tier-only baseline's own error) is doing on each test set before trusting
# either story.
for train, test, label in [(r_train, r_test, "naive random"), (g_train, g_test, "grouped")]:
    tier_pred = test["position_tier"].map(tier_mean_full).fillna(global_mean_full)
    tier_mse_split = mean_squared_error(test["ctr"], tier_pred)
    print(f"{label:15s} -- tier-only baseline MSE on this test set: {tier_mse_split:.8f}")

naive random    -- tier-only baseline MSE on this test set: 0.00001829
grouped         -- tier-only baseline MSE on this test set: 0.00003702


**Results: absolute metrics tell the honest story; the ratio metric almost hides it.** R² and MAE
both get meaningfully worse moving from the naive to the grouped split (R² 0.177 → 0.146, MAE
0.0023 → 0.0032) — the real cost of evaluating on 9 clients the model has genuinely never seen,
instead of a test set where 40 of the 44 clients also had rows in training. But "% variance
reduction vs. tier baseline" barely moves (14.46% → 15.17%)  and
that's not a contradiction, it's the diagnostic explaining itself. The frozen tier-only baseline's
own error roughly **doubles** on the grouped test clients too (MSE 0.0000181 → 0.0000370)
 almost exactly the same proportional jump as the model's own MSE (~2.03x, back-computed from the
table above). Both the model and its comparison baseline get about equally harder to predict on
clients neither has priced in before, so a ratio metric largely cancels that shared difficulty out,
even while the absolute numbers show the real cost plainly. That's a property of ratio-based
metrics generally  and exactly why every comparison table since w05 has reported R², MAE, and the
ratio side by side rather than leaning on any one alone: trusting the ratio metric here would have
made the naive split's leakage look almost harmless.

The mechanism underneath all of it: **40 of the 44 clients had rows on both sides of the naive
split.** The model, and the frozen baseline's own group-means, had already seen most of these
clients — evaluating on them tests "does this remember clients it just trained on," not "would
this generalize to a brand-new client." The grouped split's 9 clients are the only genuinely unseen
ones in this notebook, and that's the number worth trusting going forward.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**The attack checklist from `hunting-leakage-and-validating`, run against the actual final
feature set** (`avg_position_month`, `impressions_month`, `ga4_engaged_sessions_month`,
`sessions_ai_month`, `scroll_events_month`, `has_ga4_month`, `content_type`) — not a re-statement
of Week 3's, a fresh pass on what's actually in the model today.

- **Timeline drawn — mostly clean, one honest caveat.** No feature is derived from `clicks_month`
  or `ctr`. But unlike w03's row-level contract (features "knowable at report time" on a given
  day), this model aggregates everything to a single March sum — features and label share the
  *same* month, not "features strictly before the label window." That's not leakage (nothing here
  is computed *from* the label), but it does mean this model answers "how does this month's
  position/engagement associate with this month's CTR," not "can this forecast next month's CTR."
  Worth saying in plain terms rather than letting "predicts" imply forecasting it doesn't do.
- **No label-derived/sibling columns** — confirmed by re-running w03's exact style of check below:
  add `clicks_month` (the label's own numerator) back into the final feature set and watch the
  score jump. If it doesn't jump, the harness itself is broken.
- **No product flags as features** — standing rule since w04, still holds: `health_score`,
  `priority_score`, `action_type`, `needs_ctr_fix`, `is_quick_win`, `refresh_tier` never appear.
- **Population selection disclosed** — `impressions_month >= 100`, `avg_position_month >= 1`,
  published/not-deleted. These filter on exposure volume, not on `clicks_month` or `ctr`
  themselves, so they don't peek at the outcome — but they are a choice, stated here per the
  skill's own instruction to disclose rather than bury it.
- **Split grouped by the repeating entity** — Section 2's before/after table is exactly this check,
  with the gap it produces reported rather than hidden.
- **Base rate printed next to every metric** — the regression analogue of "majority class" is the
  frozen tier-only baseline; every comparison table since w05 has carried a `base_rate_mean_ctr`
  column next to every model's score, not just the winner's.
- **Top feature importance sanity-checked, not celebrated** — `ga4_engaged_sessions_month` came out
  as the single largest permutation-importance driver in w05 (0.193). Checked against the w03
  contract rather than assumed clean: it's same-day on-page behavior from GA4, a different data
  source than GSC clicks, so it's correlated with CTR through shared page quality, not circularly
  derived from the label.
- **Metrics recomputed out-of-fold** — every R²/MAE/variance-reduction number since w05 has been
  test-set-only; nothing here has been read off training data.
- **Sealed holdout — still untouched, and worth restating plainly.** The June 2026 `_sample` file
  has never been queried in any lane-4 notebook. Everything reported so far, including this
  notebook, is a validation-fold number on the March panel — real and out-of-sample relative to
  *this model's own training rows*, but not the same thing as a final number checked once against
  the sealed set. That distinction is easy to blur in casual language, so it's stated here
  directly instead.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def make_X_with(df, extra_cols, ref_cols=None):
    cols = FEATURE_COLS + extra_cols + ["content_type"]
    X = pd.get_dummies(df[cols], columns=["content_type"])
    return X.reindex(columns=ref_cols, fill_value=0) if ref_cols is not None else X

rf_kwargs = dict(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)

# Honest: final feature set, no clicks/ctr components
honest_train_X = make_X_with(g_train, [])
honest_test_X = make_X_with(g_test, [], honest_train_X.columns)
rf_honest = RandomForestRegressor(**rf_kwargs).fit(honest_train_X, g_train["ctr"])
r2_honest = r2_score(g_test["ctr"], rf_honest.predict(honest_test_X))

# Leaked: add clicks_month back in -- the label's own numerator
leaked_train_X = make_X_with(g_train, ["clicks_month"])
leaked_test_X = make_X_with(g_test, ["clicks_month"], leaked_train_X.columns)
rf_leaked = RandomForestRegressor(**rf_kwargs).fit(leaked_train_X, g_train["ctr"])
r2_leaked = r2_score(g_test["ctr"], rf_leaked.predict(leaked_test_X))

print(f"Honest R^2 (final feature set, grouped split):              {r2_honest:.4f}")
print(f"Leaked R^2 (+ clicks_month, the label's own numerator):     {r2_leaked:.4f}")
print("If the leaked number doesn't jump sharply toward 1.0, the test harness itself is broken.")

Honest R^2 (final feature set, grouped split):              0.1449
Leaked R^2 (+ clicks_month, the label's own numerator):     0.9657
If the leaked number doesn't jump sharply toward 1.0, the test harness itself is broken.


**Confirmed: the harness catches leakage when it's there.** Honest R² (final feature set, grouped
split) = 0.146 — matches Section 2's "after" row exactly, a useful cross-check that both sections
are evaluating the same held-out clients consistently. Adding `clicks_month` back in — the label's
own numerator — jumps R² to 0.966. Not quite 1.0, because `ctr = clicks_month / impressions_month`
still requires the model to combine it correctly with `impressions_month` rather than reading the
answer off directly, but the jump from 0.146 to 0.966 is unmistakable: the harness works, and
confirms none of the seven features actually in the model are doing this.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**The boldest sentence in this project's own history:** from w02/w04, *"content_type explains
~3.4% beyond position tier alone."* Bold because it reads as a general, portable finding — and
because this notebook now has the receipts showing it isn't one.

**Rewrite, mapped to the four safe-language categories the assignment names:**

> *(observed)* Within an in-sample check on the March panel, adding `content_type` to
> `position_tier` was associated with *(measured)* a ~3.4-point reduction in explained CTR
> variance beyond tier alone. *(directional)* That's a same-panel, in-sample association, not a
> validated external effect — a later out-of-client check (w05/w06) found the `tier × content_type`
> rule scores a **negative R²** on clients it hasn't seen, and that `content_type` is nearly a
> client-level constant (the median client publishes one content type 100% of the time), which is
> a plausible reason the original association didn't survive leaving the training panel.
> *(decision-support)* Read the original 3.4% as descriptive of the fitted March panel, not as a
> standalone lever — worth further out-of-client testing before it drives a real content-type
> decision, not something to act on from this number alone.

Why each swap matters: "explains" implied a causal, portable relationship; "was associated with"
doesn't. The original sentence had no scope — it didn't say *which* panel, *in-sample or
out-of-sample* — the rewrite states both. And the original gave no next step; the rewrite ends
where every claim in this project now has to end, at what a reader should actually do with it.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_rewrite = {
    "before (bold, w02/w04, in-sample, no scope stated)":
        "content_type explains ~3.4% beyond position tier alone",
    "after (safe language: observed / measured / directional / decision-support)":
        "In an in-sample check within the March panel, adding content_type to position_tier "
        "was associated with a measured ~3.4-point reduction in explained CTR variance beyond "
        "tier alone. This is directional, same-panel evidence, not validated on held-out clients: "
        "a later out-of-client check found the tier x content_type rule scores a negative R^2 on "
        "clients it hasn't seen, and content_type turned out to be nearly a client-level constant "
        "(median client publishes one type 100% of the time). Treat the original figure as "
        "descriptive of the fitted panel, not decision-ready without further out-of-client testing.",
}
for k, v in claim_rewrite.items():
    print(f"{k}:\n{v}\n")

before (bold, w02/w04, in-sample, no scope stated):
content_type explains ~3.4% beyond position tier alone

after (safe language: observed / measured / directional / decision-support):
In an in-sample check within the March panel, adding content_type to position_tier was associated with a measured ~3.4-point reduction in explained CTR variance beyond tier alone. This is directional, same-panel evidence, not validated on held-out clients: a later out-of-client check found the tier x content_type rule scores a negative R^2 on clients it hasn't seen, and content_type turned out to be nearly a client-level constant (median client publishes one type 100% of the time). Treat the original figure as descriptive of the fitted panel, not decision-ready without further out-of-client testing.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.